In [11]:
import pandas as pd 

df1 = pd.read_csv('upi_transactions_2024.csv')
df2 = pd.read_excel('UPI_transactions.xlsx')

print("=================Df1=================")
print(df1.columns.tolist())
print(df1['transaction_status'].value_counts())

print("=================DF2=================")
print(df2.columns.tolist())
print(df2['Status'].value_counts())

=================Df1=================
['transaction id', 'timestamp', 'transaction type', 'merchant_category', 'amount (INR)', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend']
transaction_status
SUCCESS    237624
FAILED      12376
Name: count, dtype: int64
=================DF2=================
['TransactionID', 'TransactionDate', 'Amount', 'BankNameSent', 'BankNameReceived', 'RemainingBalance', 'City', 'Gender', 'TransactionType', 'Status', 'TransactionTime', 'DeviceType', 'PaymentMethod', 'MerchantName', 'Purpose', 'CustomerAge', 'PaymentMode', 'Currency', 'CustomerAccountNumber', 'MerchantAccountNumber']
Status
Success    16000
Failed      4000
Name: count, dtype: int64


In [13]:
import pandas as pd


df1 = pd.read_csv('upi_transactions_2024.csv')
df2 = pd.read_excel('UPI_transactions.xlsx')

#rename the column for Final merged schema
df1.rename(columns={
   'transaction id': 'transaction_id',
    'amount (INR)': 'amount',
    'transaction_status': 'status',
    'network_type': 'payment_mode'
},inplace = True)

#deleting irrelevant columns
df1.drop(columns=['fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend'], inplace=True)

#standardization of the status column to Upper case 
df1['status'] = df1['status'].str.upper().str.strip()

#Adding missing column as null
for col in ['city', 'gender', 'purpose']:
    df1[col] = None
    
#Adding source label 
df1['source_file'] = 'primary'


#checking the final output 
print(df1.shape)
print(df1.columns.tolist())
print(df1['status'].value_counts())
print(df1.isnull().sum())

(250000, 17)
['transaction_id', 'timestamp', 'transaction type', 'merchant_category', 'amount', 'status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'payment_mode', 'city', 'gender', 'purpose', 'source_file']
status
SUCCESS    237624
FAILED      12376
Name: count, dtype: int64
transaction_id             0
timestamp                  0
transaction type           0
merchant_category          0
amount                     0
status                     0
sender_age_group           0
receiver_age_group         0
sender_state               0
sender_bank                0
receiver_bank              0
device_type                0
payment_mode               0
city                  250000
gender                250000
purpose               250000
source_file                0
dtype: int64


In [17]:
df1.rename(columns={'transaction type': 'transaction_type'}, inplace=True)
print(df1.columns.tolist())  # confirm

['transaction_id', 'timestamp', 'transaction_type', 'merchant_category', 'amount', 'status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'payment_mode', 'city', 'gender', 'purpose', 'source_file']


In [19]:
df1.to_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned/df1_cleaned.csv', index=False)
print("DF1 saved.")

DF1 saved.


In [20]:
df2 = pd.read_excel('UPI_transactions.xlsx')

# 1. Combine date + time into timestamp
df2['timestamp'] = pd.to_datetime(df2['TransactionDate'].astype(str) + ' ' + df2['TransactionTime'].astype(str))

# 2. Rename to canonical schema
df2.rename(columns={
    'TransactionID': 'transaction_id',
    'Amount': 'amount',
    'BankNameSent': 'sender_bank',
    'BankNameReceived': 'receiver_bank',
    'TransactionType': 'transaction_type',
    'Status': 'status',
    'DeviceType': 'device_type',
    'PaymentMode': 'payment_mode',
    'City': 'city',
    'Gender': 'gender',
    'Purpose': 'purpose'
}, inplace=True)

# 3. Drop columns
df2.drop(columns=['TransactionDate', 'TransactionTime', 'RemainingBalance',
                  'CustomerAge', 'MerchantName', 'PaymentMethod',
                  'Currency', 'CustomerAccountNumber', 'MerchantAccountNumber'], inplace=True)

# 4. Standardize status
df2['status'] = df2['status'].str.upper().str.strip()

# 5. Add missing columns as NULL
for col in ['merchant_category', 'sender_age_group', 'receiver_age_group', 'sender_state']:
    df2[col] = None

# 6. Source tag
df2['source_file'] = 'secondary'

# 7. Check
print(df2.shape)
print(df2.columns.tolist())
print(df2['status'].value_counts())
print(df2.isnull().sum())

(20000, 17)
['transaction_id', 'amount', 'sender_bank', 'receiver_bank', 'city', 'gender', 'transaction_type', 'status', 'device_type', 'purpose', 'payment_mode', 'timestamp', 'merchant_category', 'sender_age_group', 'receiver_age_group', 'sender_state', 'source_file']
status
SUCCESS    16000
FAILED      4000
Name: count, dtype: int64
transaction_id            0
amount                    0
sender_bank               0
receiver_bank             0
city                      0
gender                    0
transaction_type          0
status                    0
device_type               0
purpose                   0
payment_mode              0
timestamp                 0
merchant_category     20000
sender_age_group      20000
receiver_age_group    20000
sender_state          20000
source_file               0
dtype: int64


In [22]:
df2.to_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned/df2_cleaned.csv', index=False)
print("DF2 saved.")

DF2 saved.


In [23]:
# Reorder DF2 columns to match DF1
df2 = df2[df1.columns]

# Merge
merged = pd.concat([df1, df2], ignore_index=True)

# Verify
print(merged.shape)
print(merged['status'].value_counts())
print(merged['source_file'].value_counts())

(270000, 17)
status
SUCCESS    253624
FAILED      16376
Name: count, dtype: int64
source_file
primary      250000
secondary     20000
Name: count, dtype: int64


In [24]:
merged.to_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned\merged_upi_transactions.csv', index=False)
print("Merged saved.")

Merged saved.


In [25]:
df_check = pd.read_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned\merged_upi_transactions.csv')
print(df_check.shape)

(270000, 17)


C:\Users\abhijeet\AppData\Local\Temp\ipykernel_18348\3659532227.py:1: DtypeWarning: Columns (3,6,7,8,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned\merged_upi_transactions.csv')


In [1]:
import pandas as pd

df = pd.read_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned\merged_upi_transactions.csv')

# Fix timestamp format
df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True)
df['timestamp'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Save back to MySQL uploads folder
df.to_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\merged_upi_transactions.csv', index=False)
print("Done.")
print(df['timestamp'].head(3))

C:\Users\abhijeet\AppData\Local\Temp\ipykernel_27328\1929442733.py:3: DtypeWarning: Columns (3,6,7,8,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\Data Analyst Projects\UPI Payment Failure Analysis\Datasets\Kaggle\Cleaned\merged_upi_transactions.csv')


Done.
0    2024-10-08 15:17:00
1    2024-04-11 06:56:00
2    2024-04-02 13:27:00
Name: timestamp, dtype: object


In [2]:
print(df['timestamp'].head(10))

0    2024-10-08 15:17:00
1    2024-04-11 06:56:00
2    2024-04-02 13:27:00
3    2024-01-07 10:09:00
4    2024-01-23 19:04:00
5    2024-10-07 22:32:00
6    2024-02-08 10:25:00
7    2024-10-27 18:47:00
8    2024-11-21 09:39:00
9    2024-11-11 15:58:00
Name: timestamp, dtype: object
